In [40]:
# lets do chunking by loading documents

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [51]:
import re
from typing import List, Dict

def chunk_markdown_universal(markdown_text: str, doc: Document) -> List[Document]:
    """
    Universal markdown chunker that supports:
    - ### **Heading**
    - **1. Heading**, **3.1. Heading**
    - **Disclaimer:**, **Acknowledgment:**
    """

    heading_pattern = re.compile(
        r"(###\s+\*\*.*?\*\*"
        r"|\*\*\d+(?:\.\d+)*\.\s+.*?\*\*"
        r"|\*\*[A-Z][^*]{2,}?:\*\*)"
    )

    parts = heading_pattern.split(markdown_text)

    chunks = []
    chunk_id = 1

    for i in range(1, len(parts), 2):
        raw_title = parts[i].strip()
        content = parts[i + 1].strip()

        clean_title = (
            raw_title
            .replace("###", "")
            .replace("**", "")
            .strip()
        )

        chunks.append({
            "chunk_id": chunk_id,
            "title": clean_title,
            "content": clean_title+ "\n" + content
        })
        chunk_id += 1
    # convert chunks into Doccument
    # also append doc metdata to chunck
    chunks = [
        Document(
            page_content=chunk["content"], 
            metadata={**doc.metadata,
            "chunk_title": chunk["title"], 
            "chunk_id": chunk["chunk_id"]}) for chunk in chunks]
    return chunks

In [52]:
DATA_DIRECTORY = "../generated_data"

In [53]:
# lets create a loader
loader = DirectoryLoader(
    path=DATA_DIRECTORY,
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
    use_multithreading=True
)

docs = loader.load()

100%|██████████| 117/117 [00:00<00:00, 783.56it/s]


In [54]:
from pathlib import Path
def chunk_doc(doc: Document) -> List[Document]:
    """
    Convert one loaded Document into multiple chunks
    """
    source = doc.metadata.get("source", "unknown")
    title = Path(source).stem.replace("_", "-")
    doc.metadata.update({
        "title": title,
        "department": "HR",
        "company":"ABC Corp",
        "country": "India"
    })
    #print(doc.page_content)
    chunks = chunk_markdown_universal(doc.page_content, doc=doc)
    return chunks

In [55]:
all_chunks: List[Document] = []
for doc in docs:
    chunks = chunk_doc(doc)
    # create document with chunk and extend
    all_chunks.extend(chunks)

In [56]:
all_chunks[11].metadata

{'source': '..\\generated_data\\AttendancePolicy.md',
 'title': 'AttendancePolicy',
 'department': 'HR',
 'company': 'ABC Corp',
 'country': 'India',
 'chunk_title': '2. Scope',
 'chunk_id': 3}

In [57]:
all_chunks[11].page_content

'2. Scope\nThis policy applies to all employees, contractors, consultants, interns, and third-party personnel associated with Acme Corp, regardless of role, designation, or employment type.'

In [59]:
# max size of page_content in all_chunks
max_len = max(len(chunk.page_content) for chunk in all_chunks)
max_len

367